# 05 · Train, test, holdout with the TabPFN API (the playground recipe)

The same steps as the Prior Labs playground, on the public, playground-ready table
`data/tabpfn_ready/kxbtc15m_features.csv`. One change matters: the split is **by time**. A random
`train_test_split` would let the model learn from the future.

| split | period | used for |
|---|---|---|
| train | Jun 7 – Aug 2 | fitting |
| test | Aug 3 – Sep 6 | picking the trading margin, once |
| holdout | Sep 7 – Oct 3 | scored once, at the end |

Needs `TABPFN_TOKEN` (free at https://ux.priorlabs.ai). Only public data is sent to the API.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tabtrader import plots
plots.style()
pd.set_option("display.width", 140, "display.max_columns", 30)

In [2]:
import os, tabpfn_client
from tabpfn_client import TabPFNClassifier
tabpfn_client.set_access_token(os.environ["TABPFN_TOKEN"])

df = pd.read_csv(ROOT / "data" / "tabpfn_ready" / "kxbtc15m_features.csv")
features = [c for c in df.columns if c not in ("decision_ts", "split", "market", "bid", "ask", "target")]
train, test, holdout = (df[df.split == s] for s in ("train", "test", "holdout"))
print({s: len(x) for s, x in zip(("train", "test", "holdout"), (train, test, holdout))}, len(features), "features")

{'train': 5937, 'test': 5495, 'holdout': 4191} 24 features


In [3]:
model = TabPFNClassifier(model_path="v3.5_default", n_estimators=8)
model.fit(train[features], train["target"])
p_test = model.predict_proba(test[features])[:, 1]
p_hold = model.predict_proba(holdout[features])[:, 1]

In [4]:
from tabtrader.evaluate import metrics
from tabtrader.holdout import trade, _trade_frame
rows = {}
for name, pt, ph in [("TabPFN-3.5 (API)", p_test, p_hold), ("Market price", test.p_mid.values, holdout.p_mid.values)]:
    rows[name] = {**{f"holdout {k}": v for k, v in metrics(holdout.target, ph).items() if k != "n"},
                  **trade(_trade_frame(test), _trade_frame(holdout), pt, ph)}
pd.DataFrame(rows).T

,holdout logloss,holdout brier,holdout auc,holdout ece,margin_c,trades,c_per_ct,ci95,win_rate
TabPFN-3.5 (API),0.47732,0.157878,0.851352,0.020478,3,509,-1.36,"[-5.16, 1.92]",0.658
Market price,0.473952,0.157051,0.852487,0.020596,0.0,0.0,NaN,NaN,NaN


### The same split on my strategy's signals (local TabPFN-3.5, private features)

Train scores are out-of-fold; the test period sets each filter's trade rule; the holdout is the weeks I traded live.

In [5]:
from tabtrader import bankroll
d = pd.read_parquet(ROOT / "data" / "strategy" / "bluf_predictions.parquet")
rows = ["TabPFN-3.5 + Kelly", "TabPFN-3.5", "Live gate", "Random forest", "Logistic regression", "Every signal"]
pd.DataFrame({s: {m: r["final"] for m, r in bankroll.run_models(d, rows, s).items()}
              for s in ("train", "test", "holdout")}).round(2)

,train,test,holdout
TabPFN-3.5 + Kelly,7.89,8179.17,2778.63
TabPFN-3.5,0.10,11806.27,1987.93
Live gate,49.96,1275.27,2257.99
Random forest,0.00,12778.41,2210.26
Logistic regression,0.03,8806.70,923.83
Every signal,0.00,79.83,1961.82
